# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [74]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_A PI_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'Minimax-M2.5:cloud'
openai = OpenAI(base_url="http://localhost:11434/v1", api_key='api_key')

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [2]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [19]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [20]:
print(str(get_links_user_prompt("https://edwarddonner.com" ))) # print the first 500 characters of the user prompt to check it looks right           


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [75]:
def select_relevant_links(url):
    
    response = openai.chat.completions.create(
        model= MODEL,
     messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": str(get_links_user_prompt(url))}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [76]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services/curriculum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services/program', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'services/program',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'services/program', 'url': 'https://edwarddonner.com/outsmart/'}]}

In [77]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [78]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling Minimax-M2.5:cloud
Found 2 relevant links


{'links': [{'type': 'main website', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [79]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Minimax-M2.5:cloud
Found 18 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co'},
  {'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'github', 'url': 'https://gith

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [80]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [81]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling Minimax-M2.5:cloud
Found 17 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-9B
Updated
6 days ago
•
693k
•
557
Qwen/Qwen3.5-35B-A3B
Updated
8 days ago
•
1.09M
•
1.02k
Qwen/Qwen3.5-0.8B
Updated
5 days ago
•
346k
•
309
Lightricks/LTX-2.3
Updated
2 days ago
•
119k
•
309
Qwen/Qwen3.5-4B
Updated
6 days ago
•
283k
•
284
Browse 2M+ models
Spaces
Running
on
Zero
Featured
411
Omni Video Factory
🏆
411
text to video, image to video, video extend
Running
on
Zero
MCP
1.13k
Wan2.2 14B Preview
🐌
1.13k
generate a video from an image with a text prompt
Running
on
Zero
133
OBLITERATUS
💥
133
One-click mode

In [82]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [83]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [84]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Minimax-M2.5:cloud
Found 18 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-9B\nUpdated\n6 days ago\n•\n693k\n•\n557\nQwen/Qwen3.5-35B-A3B\nUpdated\n8 days ago\n•\n1.09M\n•\n1.02k\nQwen/Qwen3.5-0.8B\nUpdated\n5 days ago\n•\n346k\n•\n309\nLightricks/LTX-2.3\nUpdated\n2 days ago\n•\n119k\n•\n309\nQwen/Qwen3.5-4B\nUpdated\n6 days ago\n•\n283k\n•\n284\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n411\nOmni Video Factory\n🏆\n411\ntext to video, i

In [85]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [86]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Minimax-M2.5:cloud
Found 18 relevant links


# Hugging Face

## The AI Community Building the Future

---

### About Hugging Face

Hugging Face is the premier collaboration platform where the machine learning community comes together to build the future of AI. The company provides a comprehensive ecosystem for creating, discovering, and collaborating on machine learning models, datasets, and applications.

---

### What We Offer

**🤖 Models**
Access to over 2 million pre-trained models across virtually every AI task—from text generation to computer vision. Browse models by task, parameters, library, and more.

**📊 Datasets**
Explore more than 500,000 datasets for training and benchmarking machine learning models. The platform supports diverse data types and research needs.

**🚀 Spaces**
Discover and run over 1 million AI applications. From text-to-video generators to image editing tools, Spaces showcases the creative possibilities of AI.

---

### Platform Features

| Feature | Description |
|---------|-------------|
| **Collaboration** | Host and collaborate on unlimited public models, datasets, and applications |
| **Open Source Stack** | Build faster with HF's open source tools and libraries |
| **Multi-Modality** | Support for text, image, video, audio, and 3D AI |
| **Portfolio Building** | Share your work with the world and build your ML profile |

---

### Featured Applications

The platform hosts innovative AI applications including:

- **Omni Video Factory** — Text-to-video, image-to-video, and video extension
- **Wan2.2 14B Preview** — Generate videos from images with text prompts
- **Qwen Image Multiple Angles 3D Camera** — Change photo camera angles with AI
- **FireRed Image Edit** — AI-powered image editing capabilities

---

### Enterprise Solutions

Hugging Face provides paid Compute and Enterprise solutions for teams and organizations looking to accelerate their machine learning initiatives. These solutions include:

- Team collaboration tools
- Enterprise-grade infrastructure
- Dedicated support
- Scalable compute resources

---

### Join the Community

Whether you're a researcher, developer, or AI enthusiast, Hugging Face offers the tools and community support you need to advance your machine learning journey.

**Get Started Today:**

- 🌐 Website: [huggingface.co](https://huggingface.co)
- 📚 Explore Documentation
- 💻 Browse Models, Datasets, and Spaces
- 🏢 Learn About Enterprise Solutions

---

*Note: This brochure was compiled from publicly available information. For the most current details about company culture, career opportunities, and customer case studies, please visit huggingface.co directly.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [87]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [89]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Minimax-M2.5:cloud
Found 12 relevant links


# Hugging Face

## The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the collaboration platform for the machine learning community. Founded with a mission to build an open and ethical AI future, the company has become the central hub where machine learning engineers, scientists, and end users come together to learn, collaborate, and share their work.

The platform serves as a central place where anyone can share, explore, discover, and experiment with open-source machine learning. Hugging Face empowers the next generation of ML professionals to innovate, collaborate, and contribute to the advancement of artificial intelligence in a community-driven environment.

---

## What We Offer

### The Hub — Your All-in-One ML Platform

Hugging Face provides a comprehensive ecosystem for machine learning development:

- **Models**: Access over 2 million pre-trained models for various AI tasks
- **Spaces**: Discover and run over 1 million AI applications and demos
- **Datasets**: Explore more than 500,000 datasets for training and research
- **Community**: Connect with a thriving global community of ML practitioners

### Multi-Modality Support

Build across every type of AI:

- Text generation and processing
- Image generation and editing
- Video synthesis
- Audio and speech
- 3D modeling

### Open Source Foundation

Leverage the HF open-source stack to move faster and build better AI solutions. The platform integrates seamlessly with popular ML libraries and frameworks.

---

## Enterprise Solutions

Hugging Face offers scalable solutions for teams and organizations:

### Team Plans

- Starting at $20/user/month
- Enhanced collaboration tools
- Shared resources and repositories

### Enterprise Solutions

For organizations requiring advanced capabilities, Hugging Face Enterprise provides:

- **Security**: Single Sign-On (SSO) integration and advanced security policies
- **Data Control**: Region selection, audit logs, and comprehensive token management
- **Access Management**: Resource groups with granular access control
- **Storage**: 1 TB private storage per member (additional at $25/month per TB)
- **Compute**: Advanced compute options including ZeroGPU with 5x quota boost
- **Analytics**: Unified dashboard for tracking repository usage
- **Billing**: Managed billing with yearly commitment options
- **Support**: Priority support from the Hugging Face team

---

## Join the Community

Whether you're a researcher, developer, data scientist, or AI enthusiast, Hugging Face provides the tools and community you need to advance your machine learning journey.

### Build Your Portfolio

Share your work with the world, showcase your ML projects, and build your professional profile in the AI community.

### Collaborate Without Limits

Host and collaborate on unlimited public models, datasets, and applications with the peace of mind that comes from enterprise-grade infrastructure.

---

## Get Started Today

**Visit**: [huggingface.co](https://huggingface.co)

- Explore 2M+ models
- Browse 1M+ applications
- Access 500k+ datasets
- Join the AI revolution

---

*Together, we're building an open and ethical AI future.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>